In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import mplhep as hep
hep.style.use("CMS")
from coffea import util
import itertools
import os, sys
import glob
import copy
import uproot

import mplhep as hep
hep.style.use("CMS")

sys.path.append('../python/')
import functions

In [ ]:
IOV = '2017'

In [ ]:
# qcd16 = util.load(f'../outputs/scale/QCD_2016all.coffea')
# qcd17 = util.load(f'../outputs/scale/QCD_2017.coffea')
# qcd18 = util.load(f'../outputs/scale/QCD_2018.coffea')


# ttbar17 = util.load(f'../outputs/scale/TTbar_2017.coffea')
# ttbar16 = util.load(f'../outputs/scale/TTbar_2016all.coffea')
# ttbar18 = util.load(f'../outputs/scale/TTbar_2018.coffea')


data17B = util.load(f'../outputs/JetHT_2017B.coffea')
data17C = util.load(f'../outputs/JetHT_2017C.coffea')
data17D = util.load(f'../outputs/JetHT_2017D.coffea')
data17E = util.load(f'../outputs/JetHT_2017E.coffea')
# data17F = util.load(f'../outputs/JetHT_2017F_bkgest.coffea')
# data16 = util.load(f'../outputs/JetHT_2016APVB.coffea')
# data18 = util.load(f'../outputs/scale/JetHT_2018.coffea')


# qcd17 = util.load(f'../outputs/scale/QCD_2017.coffea')
# ttbar17 = util.load(f'../outputs/scale/TTbar_2017.coffea')
# data17 = util.load(f'../outputs/scale/JetHT_2017.coffea')

# qcd16 = util.load(f'../outputs/scale/QCD_2016all.coffea')
# ttbar16 = util.load(f'../outputs/scale/TTbar_2016all.coffea')
# data16 = util.load(f'../outputs/scale/JetHT_2016all.coffea')

## Cutflow

In [ ]:
data = {}
cuts = []

for cut in data17B['cutflow'].keys():
    if 'sumw' in cut: continue
        
    evts = data17B['cutflow'][cut] + data17C['cutflow'][cut] + data17D['cutflow'][cut] + data17E['cutflow'][cut] #+ data17F['cutflow'][cut]
    data[cut] = evts
    cuts.append(evts)

# total = 38619202


In [ ]:
df = pd.DataFrame(data=cuts, index=data.keys())
df

In [ ]:
table1 = '''
\\begin{table}
    \\begin{center}
        \\rowcolors{2}{gray!15}{white}
        \\begin{tabular}{ |p{6cm}||p{3cm}|  } \\hline 
            \\textbf{Cut} & \\textbf{Events} \\\\ \\hline
            All Events                                  &  '''+'{0:0,.0f}'.format(data['all events'])+''' \\\\
            Trigger                                     &  '''+'{0:0,.0f}'.format(data['trigger'])+''' \\\\ 
            $HT > 1400$                                 &   '''+'{0:0,.0f}'.format(data['htCut'])+''' \\\\ 
            METfilter                                   &   '''+'{0:0,.0f}'.format(data['metfilter'])+''' \\\\ 
            $\\ge 1$ AK8 jet passing JetID               &   '''+'{0:0,.0f}'.format(data['jetid'])+''' \\\\ 
            $\\ge 1$ AK8 $p_T > 400$ GeV, $|y| < 2.4$    &   '''+'{0:0,.0f}'.format(data['jetkincut'])+''' \\\\ 
            $>2$ AK8 Jets                               &   '''+'{0:0,.0f}'.format(data['twoFatJets'])+''' \\\\ 
            %$\\geq 1$ ttbar candidate                   &   '''+'{0:0,.0f}'.format(data['oneTTbar'])+''' \\\\
            $|\\Delta\\varphi| > 2.1$                     &   '''+'{0:0,.0f}'.format(data['dPhiCut'])+''' \\\\ 
            \\hline
        \\end{tabular}
        \\caption{Cutflow for blinded JetHT events for 2016, 2017 and 2018 datasets.}
        \\label{tab:cutflow}
    \\end{center}
\\end{table}

'''

In [ ]:
print(table1)

## Weights

In [ ]:
wgts = {}
wgts['2018'] = ttbar18['weights']
wgts['2017'] = ttbar17['weights']
wgts['2016'] = ttbar16['weights']

wdata = {}
wdata2 = {}

for year in ['2016', '2017', '2018']:
    
    wdata[year] = {}
    wdata2[year] = {}
    nom = wgts[year]['nominal']
    
    for wgt in wgts[year]:
        
#         print(wgt)
        
        
        if not 'nominal' in wgt:
            
            try:

                wdata2[year][wgt] = wgts[year][wgt]/nom

                wdata[year][wgt] = np.abs(1-wgts[year][wgt]/nom) #np.abs(1-np.abs(wgts[wgt]/nom))
            except:
                continue


In [ ]:
for year in wdata2.keys():
    print(year)
    
    for key in wdata2[year].keys():
    
        print(key, '{0:0.3f}'.format(wdata2[year][key]), '{0:0.3f}'.format(wdata[year][key]))

In [ ]:
table2 = '''
\\begin{table}
    \\begin{center}
        \\begin{tabular}{ |p{1cm}||p{3cm}| |p{3cm}| } \\hline 
            \\textbf{Year} & \\textbf{Systematic} & \\textbf{up/down} \\\\ \\hline
              2016   & PileUp       &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['pileupUp'], wdata['2016']['pileupDown'])+''' \\\\
                     & PDF          &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['pdfUp'],    wdata['2016']['pdfDown'])+''' \\\\ 
                     & $Q^2$        &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['q2Up'],     wdata['2016']['q2Down'])+''' \\\\ 
                     & btag         &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['btagUp'],   wdata['2016']['btagDown'])+''' \\\\ 
                     & JES          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['jesUp'],    wdata['2016']['jesDown'])+''' \\\\ 
                     & JER          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2016']['jerUp'],    wdata['2016']['jerDown'])+''' \\\\
                     & L1 Prefiring &   '''+'+{0:0.2f}/+{1:0.2f}'.format(wdata['2017']['prefiringUp'],    wdata['2017']['prefiringDown'])+''' \\\\
              \\hline    
              2017   & PileUp       &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['pileupUp'], wdata['2017']['pileupDown'])+''' \\\\
                     & PDF          &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['pdfUp'],    wdata['2017']['pdfDown'])+''' \\\\ 
                     & $Q^2$        &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['q2Up'],     wdata['2017']['q2Down'])+''' \\\\ 
                     & btag         &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['btagUp'],   wdata['2017']['btagDown'])+''' \\\\ 
                     & JES          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['jesUp'],    wdata['2017']['jesDown'])+''' \\\\ 
                     & JER          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2017']['jerUp'],    wdata['2017']['jerDown'])+''' \\\\
                     & L1 Prefiring &   '''+'+{0:0.2f}/+{1:0.2f}'.format(wdata['2017']['prefiringUp'],    wdata['2017']['prefiringDown'])+''' \\\\
              \\hline
              2018   & PileUp       &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['pileupUp'], wdata['2018']['pileupDown'])+''' \\\\
                     & PDF          &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['pdfUp'],    wdata['2018']['pdfDown'])+''' \\\\ 
                     & $Q^2$        &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['q2Up'],     wdata['2018']['q2Down'])+''' \\\\ 
                     & btag         &   '''+'+{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['btagUp'],   wdata['2018']['btagDown'])+''' \\\\ 
                     & JES          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['jesUp'],    wdata['2018']['jesDown'])+''' \\\\ 
                     & JER          &   '''+'-{0:0.2f}/-{1:0.2f}'.format(wdata['2018']['jerUp'],    wdata['2018']['jerDown'])+''' \\\\
                     & HEM          &   '''+'-{0:0.2f}'.format(wdata['2018']['hem'])+''' \\\\

               \\hline
        \\end{tabular}
        \\label{tab:syst}
        \\caption{Systematic uncertainties for 2016, 2017, 2018 MC TTbar datasets}
    \\end{center}
\\end{table}

'''

In [ ]:
print(table2)

In [ ]:
'+{0:0.2f}/-{1:0.2f}'.format(wdata['pileupUp'],wdata['pileupDown'])